# Homework 4: Hand-eye coordination (3-4 hours)

## 1. Introduction (~10 min)

In this homework, we will focus on finding the 3D point that corresponds to a 2D point in an image. 

In the theory lectures, the pinhole model was discussed. We recommend to make yourself very familiar with this model for image formation, as we will use it extensively. In particular you should be aware of the different frames involved and how you can convert between them. The figure below contains a schematic overview of all frames involved:

![](img/pinhole-frames.png)






## 2. Installation (~15 min) 


- make sure you have activated the conda environment
- If you don't have open3d installed yet, you can do it by running `pip install open3d` from your command line or in this notebook (use `!` to run terminal commands)
Normally you would execute these commands in your terminal, but we can also execute them in a notebook by prepending a `!`. 

<div class="alert alert-block alert-info"> <b>Task 1:</b> Uncomment and run the lines below to install open3d and update airo-mono.</div>




In [ ]:
!current_env=$(conda info --envs | grep "*" | awk '{print $1}'); if [ "$current_env" = "irm" ]; then echo "The current active Conda environment is 'irm'. You can run the next cell."; else echo "The current active Conda environment is not 'irm'. Make sure to run this notebook with your irm conda env (conda activate irm)"; fi

In [1]:
!pip install open3D

If you have problems installing Open3D, or get problems with Open3D along the way, downgrade numpy by uncommenting the cell below and running it.

In [ ]:
#!pip install numpy~=1.26 # open3D requires numpy < 2.0 for now
#!pip uninstall open3D -y
#!pip install open3D

In [4]:
!pip install pyrealsense2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 9.4 MB/s  0:00:01m 9.1 MB/s eta 0:00:01


To work with a real camera in the practical session, you need to install the SDK and the python bindings for the realsense cameras. 

<div class="alert alert-block alert-info"> <b>Task 2:</b> Install librealsense2 and pyrealsense2 by following the the steps 
<a href="https://github.com/airo-ugent/airo-mono/blob/main/airo-camera-toolkit/airo_camera_toolkit/cameras/realsense/realsense_installation.md">here</a> (you don't need to do the optional steps). Run the cell below to test your installation.
 </div>

Make sure you can run the command `realsense-viewer` in your terminal.

You don't need `pyrealsense2` for the homework, but you can run the cell below to test your installation. 
(You might need to restart your notebook.)

In [5]:
try:
    import pyrealsense2 as rs 
except ImportError:
    print("Install the librealsense2 and pyrealsense2 first")

## 3. Exploring the data (~45 min)

We have collected some images for you using a [Zed2i]() camera and we will use these images to complete this homework.


 The Zed camera contains actually two RGB sensors, which we will refer to as the `left_camera` and `right_camera`. These are placed 12cm apart and by combining them, the camera can internally create pointclouds and depth images. An illustration of both sensor frames can be found below:

![stereo](https://i.imgur.com/KIz1YpU.png[/img])
 <!-- source: https://www.researchgate.net/profile/Luiz-Goncalves-4/publication/325854193/figure/fig2/AS:639242699026439@1529418744804/The-operation-principle-of-the-ZED-camera.png -->



 The data used in this notebook is available at:
https://ugentbe-my.sharepoint.com/:f:/g/personal/victorlouis_degusseme_ugent_be/Emy4bRn2FJdBvWOtEnu2Jx8BSCaLEM3NbIdbwsQ-TdAQrg?e=au3xRk


<div class="alert alert-block alert-info"> <b>Task 3:</b> Download and unzip to the same directory as this notebook. Your folder structure should look like this:
</div>

```
../
    homework_4.ipynb
    data_0000/
        ...
```
Run the next cell to verify if the data has been set up correctly:

In [ ]:
import os
import numpy as np

DATA_DIR = os.path.join(os.getcwd(), 'data_0000')
data_dir = DATA_DIR
print(f"data directory: {DATA_DIR}")
assert os.path.exists(DATA_DIR) and len(os.listdir(DATA_DIR)) == 7, "Make sure you have downloaded the data and placed it in the correct directory"

We wil explore the data that has been collected for you. Start by running the next cell, which will create variables that point to the paths of all files.


In [ ]:
intrinsics_path = os.path.join(data_dir, "intrinsics.json")
image_left_path = os.path.join(data_dir, "image_left.png")
image_right_path = os.path.join(data_dir, "image_right.png")
depth_map_path = os.path.join(data_dir, "depth_map.tiff")
pointcloud_path = os.path.join(data_dir, "pointcloud.ply")
camera_pose_path = os.path.join(data_dir, "camera_pose.json")

We will now load the intrinsics and extrinsics. 
Note that we have only a single intrinsics matrix for both the left and right camera, but two separate extrinsics matrices (as both sensors have a different location).


In [ ]:
from airo_dataset_tools.data_parsers.camera_intrinsics import CameraIntrinsics
from airo_dataset_tools.data_parsers.pose import Pose


with open(intrinsics_path, "r") as f:
    intrinsics = CameraIntrinsics.model_validate_json(f.read()).as_matrix()

with open(camera_pose_path, "r") as f:
    left_extrinsics = Pose.model_validate_json(f.read()).as_homogeneous_matrix()


# create extrinsics for the right camera, which is 12 cm to the right of the left camera
right_camera_pose_in_left_camera_frame = np.eye(4)
right_camera_pose_in_left_camera_frame[0,3] = 0.12 # 12 cm to the right of the left camera
right_extrinsics = left_extrinsics @ right_camera_pose_in_left_camera_frame 

with np.printoptions(precision=3, suppress=True):
    print("Intrinsics: \n", intrinsics)
    print("Extrinsics: \n", left_extrinsics)
    print("Extrinsics for right camera: \n", right_extrinsics)


<div class="alert alert-block alert-info"> <b>Task 4:</b> To make yourselves familiar with these matrices and what they represent, answer the following two questions:</div>

1. We have taken the base frame of the robot as world frame for the extrinsics. How far is the left camera away from the base frame? 
2. Looking at the intrinsics matrix, can you guess the image resolution?

In [ ]:
#TODO (@Students): fill in the answer to the questions about the intrinsics and extrinsics
camera_to_robot_distance = None # in meters

print(f"Camera to robot distance: {camera_to_robot_distance}")
assert camera_to_robot_distance and  camera_to_robot_distance > 0, "Make sure you have filled in the camera to robot distance"


In [ ]:
estimated_image_resolution = None # (image height, image width) in pixels

print(f"Estimated image resolution: {estimated_image_resolution}")
assert estimated_image_resolution and estimated_image_resolution[0] > 0 and estimated_image_resolution[1] > 0, "Make sure you have filled in the estimated image resolution"

Next, we will take a look at the RGB images of both the left and right sensor.

In [ ]:
import matplotlib.pyplot as plt

image_left = plt.imread(image_left_path)
image_right = plt.imread(image_right_path)

def imshow_left_right(image_left, image_right):
    plt.figure(figsize=(20, 10))
    plt.subplot(1, 2, 1)
    plt.imshow(image_left)
    plt.title("Left image")
    plt.subplot(1, 2, 2)
    plt.imshow(image_right)
    plt.title("Right image")
    plt.show()


imshow_left_right(image_left, image_right)

<div class="alert alert-block alert-info"> <b>Task 5:</b> Now verify your guess of the image resolution by looking at the shape of the image tensors.
</div>

In [ ]:
#TODO (@Students): get the resolution of the left image and check if it matches the estimated resolution based on the intrinsics matrix

image_width =  None # in pixels
image_height = None # in pixels

print(f"expected image resolution: {estimated_image_resolution}")
print(f"actual image resolution: {image_width, image_height}")
assert image_width and image_width > 0, "Make sure you have filled in the image width"
assert image_height and image_height > 0, "Make sure you have filled in the image height"

Next, we will take a look at the pointcloud provided by the ZED camera.

As you know, pointcloud is a set of points (x,y,z); possible with color information (r,g,b) for each point. 

You don't need to worry (for now) about how this pointcloud is created!

In [ ]:
import open3d as o3d

pointcloud_in_camera_left = o3d.io.read_point_cloud(pointcloud_path)
pointcloud_in_robot_base = pointcloud_in_camera_left.transform(left_extrinsics)

<div class="alert alert-block alert-info"> <b>Task 6:</b> Run the cell below and look around in the pointcloud. </div>

```
-- Mouse view control --
  Left button + drag         : Rotate.
  Ctrl + left button + drag  : Translate.
  Wheel button + drag        : Translate.
  Shift + left button + drag : Roll.
  Wheel                      : Zoom in/out.
```

**Fun excercise:** can you try to find the viewpoint of the camera?

In [ ]:
# RGB float colors
cyan = (0.0, 1.0, 1.0)  # will be used for the left camera
magenta = (1.0, 0.0, 1.0)  # will be used for the right camera
yellow = (1.0, 1.0, 0.0)  # will be used for triangulation solution


## open3D helper functions
def visualize_open3D(geometry_list):
    o3d.visualization.draw_geometries(
        geometry_list,
        zoom=0.2,
        front=[1.0, 0.0, 1.0],
        lookat=[0.0, -0.3, 0.0],
        up=[0.0,0, 1.0],
    )


def open3d_ray(direction, origin=np.array([0, 0, 0])):
    c = 0.002
    ray_mesh = o3d.geometry.TriangleMesh.create_arrow(c, 4 * c, 1.5, 5 * c)

    rotation_Z = direction
    rotation_X = np.cross(rotation_Z, np.array([0, 1, 0]))
    rotation_Y = np.cross(rotation_Z, rotation_X)
    rotation_matrix = np.column_stack([rotation_X, rotation_Y, rotation_Z])

    ray_mesh.rotate(rotation_matrix, center=(0, 0, 0))

    ray_mesh.translate(origin)

    return ray_mesh


base_frame_mesh = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.3)

width, height = image_width, image_height

camera_left_pose_mesh = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
camera_left_pose_mesh.transform(left_extrinsics)
camera_left_lines = o3d.geometry.LineSet.create_camera_visualization(
    width, height, intrinsics, np.linalg.inv(left_extrinsics), scale=0.1
)
camera_left_lines.paint_uniform_color(cyan)

camera_right_pose_mesh = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
camera_right_pose_mesh.transform(right_extrinsics)
camera_right_lines = o3d.geometry.LineSet.create_camera_visualization(
    width, height, intrinsics, np.linalg.inv(right_extrinsics), scale=0.1
)
camera_right_lines.paint_uniform_color(magenta)

visualize_open3D(
    [
        pointcloud_in_robot_base,
        base_frame_mesh,
        camera_left_pose_mesh,
        camera_left_lines,
        camera_right_pose_mesh,
        camera_right_lines,
    ]
)

## 4. Image Triangulation (2-3 hours)

Triangulation is the problem of going from corresponding 2D points (pixels) to a 3D position in the camera frame. As we already know, a point on an image is a ray in the camera frame. So we need to combine multiple rays to find a 3D point. This is what triangulation does. The principle is also illustrated in this image:
![triangulation](https://res.cloudinary.com/tbmg/c_scale,w_800,f_auto,q_auto/v1522165222/sites/tb/articles/2012/features/40438-121_fig4.jpg)
<!-- source: https://www.techbriefs.com/component/content/article/tb/pub/features/articles/14925 -->


There are three steps:

1. we need to gather the pixel coordinates that correspond to a 3D point.
2. we need to convert these pixels into rays from their camera frames and express them in the common world frame
3. we need to perform triangulation to estimate a 3D point that is as close as possible to the rays from each view.

We will tackle these steps one by one below.

### 4.1. Collect corresponding pixels.

We have provided the coordinates of the charuco board's bottomleft corner, but at the end of the homework you will have to run this again with a point of your choice.

In [ ]:
import cv2

image_left_point = (994,406)
image_right_point = (824,405)

image_left_annotated = cv2.circle(image_left, image_left_point, 15, cyan, 5)
image_right_annotated = cv2.circle(image_right, image_right_point, 15, magenta, 5)

imshow_left_right(image_left_annotated, image_right_annotated)

### 4.2. Create the rays

To create a ray from a point on the image, we need to recall the relation between a point and its image coordinates:

$$c  \left(\begin{matrix}u\\v\\1\end{matrix}\right)= K \left(\begin{matrix}X\\Y\\Z\end{matrix}\right)$$

From this, it follows that the ray is represented by the vector $(X,Y,Z)$, which can be obtained as:
$$ {}^C\!r = c' \left(\begin{matrix}X\\Y\\Z\end{matrix}\right) = K^{-1} \left( \begin{matrix}u\\v\\1\end{matrix} \right) \quad \forall  c' $$

Converting this ray to the world frame is then as simple as multiplying with the rotational part of the extrinsics matrix:
$${}^W\!r =  {}^W\! R^C  {}^C\!r $$


<div class="alert alert-block alert-info"> <b>Task 7:</b> Implement these two steps in the function below.</div>

In [ ]:
def backproject_pixel_to_ray(point: np.ndarray,intrinsics: np.ndarray, extrinsics: np.ndarray) -> np.ndarray:
    """ takes in (u,v) pixel coordinates and the intrinsics matrix and returns the ray in camera frame 
    
    point: np.ndarray of shape (2,) with (u,v) pixel coordinates
    intrinsics: np.ndarray of shape (3,3) with the intrinsics matrix
    extrinsics: np.ndarray of shape (4,4) with the extrinsics matrix (camera pose in world frame)

    returns: np.ndarray of shape (3,) with the ray in world frame
    """
    #TODO (@Students): fill in the function 
    raise NotImplementedError


left_ray_direction = backproject_pixel_to_ray(image_left_point, intrinsics,left_extrinsics)
right_ray_direction = backproject_pixel_to_ray(image_right_point, intrinsics,right_extrinsics)

left_ray_origin = left_extrinsics[:3, 3]
right_ray_origin = right_extrinsics[:3, 3]

with np.printoptions(precision=3, suppress=True):
    print("Left ray direction: ", left_ray_direction)
    print("Right ray direction: ", right_ray_direction)
    print("Left ray origin:", left_ray_origin)
    print("Right ray origin:", right_ray_origin)


 We have written some code to visualize the resulting rays on top of the pointcloud, make sure they do indeed direct towards the point you clicked on before.

In [ ]:
open3d_left_ray = open3d_ray(left_ray_direction, left_ray_origin)
open3d_left_ray.paint_uniform_color(cyan)
open3d_right_ray = open3d_ray(right_ray_direction, right_ray_origin)
open3d_right_ray.paint_uniform_color(magenta)

visualize_open3D([pointcloud_in_robot_base,
        open3d_left_ray, 
        open3d_right_ray,
        camera_left_lines,
        camera_right_lines,])

The result should look like this:

![rays example](img/rays-example.png)

T make sure you have implemented it right, gather the coordinates of a number of other points using the python script and verify that the rays are still correct!

### 4.3. Finding the best 3D point 

Now that we have the rays, we can find a point that minimizes the distance to each ray. This is called the [Least Squares method for triangulation](https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=8967077&tag=1), because we minimize the L2 ( aka squared) norm between the 3D point and the rays. 

We will go over the mathematics of this method in the practical session but we encourage you to derrive the formula yourself, it is a good excercise for working with the pinhole model. 

> hint: the orthogonal projection of a vector $u$ onto another vector $b$ can be described as $\frac{b.b^T}{\|b\|^2} u$

The formula for the 3D point estimation is as follows:

$$X^{'} = \left(\sum_{i}{I - \frac{b_i.b_i^T}{\|b_i\|^2}}\right)^{-1}\left(\sum_{i}{(I - \frac{b_i.b_i^T}{\|b_i\|^2}) O_i}\right)$$

where:
* $X^{'}$ is the point that minimizes the distance to each ray
* $O_i$ is the origin of the $i$ th camera in the world frame
* $b_i$ is the ray from the $i$ th camera in the world frame 
* $I$ is the identity matrix.

In the next function, you should implement this formula for the case with 2 views: left and right. To obtain the rays $b_i$ you can use the result of step 2. 

<div class="alert alert-block alert-info"> <b>Task 8:</b> Implement the triangulation function below.</div>

We have added code that visualizes the 3D point in the point cloud so that you can verify if it works as expected. Make sure to test with a number of different points in the images!

In [ ]:
def least_squares_triangulation(point_left: np.ndarray, point_right: np.ndarray, intrinsics: np.ndarray, left_extrinsics: np.ndarray, right_extrinsics:np.ndarray) -> np.ndarray:
    """Triangulates a 3D point from a pair of 2D points in the left and right image

    point_left: np.ndarray of shape (2,) with (u,v) pixel coordinates in the left image
    point_right: np.ndarray of shape (2,) with (u,v) pixel coordinates in the right image
    intrinsics: np.ndarray of shape (3,3) with the intrinsics matrix
    left_extrinsics: np.ndarray of shape (4,4) with the extrinsics matrix (camera pose in world frame) for the left camera
    right_extrinsics: np.ndarray of shape (4,4) with the extrinsics matrix (camera pose in world frame) for the right camera

    returns: np.ndarray of shape (3,) with the 3D point in the world frame
    """
    #TODO (@Students): implement the least squares triangulation method

    raise NotImplementedError

    
point_3d = least_squares_triangulation(image_left_point, image_right_point, intrinsics, left_extrinsics, right_extrinsics)
point_3d_in_camera_frame = np.linalg.inv(left_extrinsics) @ np.append(point_3d, 1)
point_3d_in_camera_frame = point_3d_in_camera_frame[:3] / point_3d_in_camera_frame[3]

with np.printoptions(precision=3, suppress=True):
    print(f"3D point: {point_3d}")
    print(f"3D point in camera frame: {point_3d_in_camera_frame}")

Run the visualization:

In [ ]:
point_vis = o3d.geometry.TriangleMesh.create_sphere(radius=0.01)
point_vis.paint_uniform_color(yellow)
point_vis.translate(point_3d)
visualize_open3D(
    [
        pointcloud_in_robot_base,
        point_vis,
        open3d_left_ray,
        open3d_right_ray,
        camera_left_lines,
        camera_right_lines,
    ]
)

The result should look like this:

![](img/triangulate-example.png)


We have now managed to turn our 2D pixels into a 3D position, which we can then reach with our robot! Using this, in the practical session we will learn how to grasp objects in arbitrary poses, instead of measuring and hardcoding their position as we did before!

## 5. Try it yourself!

<div class="alert alert-block alert-info"> <b>Task 9:</b> Run <b>image_viewer.py</b> and click a 3D point in the left and right image and note the pixel (u, v) coordinates. Fill these in at the start of the Triangulation section and run the code again. </div>